# Python `str.splitlines()`: Universal Newlines & `keepends`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_splitlines_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.splitlines() Tutorial](https://www.plus2net.com/python/string-splitlines.php)**.

### Learning Objectives:
1. Understand universal newline splitting across Unix (`\n`), Windows (`\r\n`), Classic Mac (`\r`), and Unicode line breaks.
2. Avoid the classic traps of `split('\n')` (spurious trailing empty items and carriage return leaks).
3. Use `keepends=True` to preserve native line boundaries for exact document reconstruction.
4. Build production text pipelines for cross-platform upload normalization, code line numbering, and grouped multiline log parsing.
5. Validate implementations with automated self-checking unit assertions.

## 1. Universal Newlines: Beyond Simple `\n`

Python's `str.splitlines()` recognizes standard ASCII control breaks (LF, CRLF, CR, VT, FF) and Unicode breaks (Line Separator `\u2028`, Paragraph Separator `\u2029`).

In [ ]:
# Universal newline handling across different OS formats
mixed_lines = "Unix Line\nWindows Line\r\nMac Line\rNext Line\x85Unicode Line\u2028Final Line"

parsed = mixed_lines.splitlines()
print("Parsed lines count:", len(parsed))
for idx, line in enumerate(parsed, 1):
    print(f"  {idx}. {line}")

assert len(parsed) == 6
assert parsed[0] == "Unix Line"
assert parsed[1] == "Windows Line"
assert parsed[2] == "Mac Line"

print("\nAll universal line-break assertions passed!")

## 2. `splitlines()` vs. `split('\n')`: The Two Critical Traps

1. **The Trailing Empty Item Trap**: `split('\n')` adds a spurious empty string `""` when text ends with a newline. `splitlines()` does not.
2. **The CRLF Leak Trap**: `split('\n')` leaves stray carriage returns `\r` at the ends of Windows lines.

In [ ]:
# Trap 1: Trailing newline
doc_with_trailing = "Header\nRow 1\nRow 2\n"

bad_split = doc_with_trailing.split("\n")
good_split = doc_with_trailing.splitlines()

print("split('\n'): ", bad_split)
print("splitlines():", good_split)
assert len(bad_split) == 4 and bad_split[-1] == ""
assert len(good_split) == 3

# Trap 2: Windows CRLF
win_lines = "Row1\r\nRow2\r\nRow3"
bad_win = win_lines.split("\n")
good_win = win_lines.splitlines()

print("bad_win[0] (contains \\r):", repr(bad_win[0]))
print("good_win[0] (clean):       ", repr(good_win[0]))
assert bad_win[0] == "Row1\r"
assert good_win[0] == "Row1"

# Empty string input
assert "".splitlines() == []
assert "".split("\n") == [""]

print("\nAll comparison trap assertions passed!")

## 3. Preserving Line Breaks with `keepends=True`

When `keepends=True`, line break characters are kept at the end of each resulting string element, enabling exact round-trip document reassembly.

In [ ]:
source_text = "First line\nSecond line\r\nThird line"

# Split while preserving exact line terminators
kept = source_text.splitlines(keepends=True)
print("Lines with line breaks preserved:")
for item in kept:
    print(" ", repr(item))

assert kept == ["First line\n", "Second line\r\n", "Third line"]

# Round-trip reassembly
reconstructed = "".join(kept)
assert reconstructed == source_text
print("\nRound-trip reassembly successful: 100% byte identical!")

## 4. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.splitlines()`:
1. Normalizing Cross-Platform HTTP Uploads into POSIX `\n`
2. Numbered Source Code Formatter
3. Log Parser with Continuation Lines (Grouping Multiline Stack Traces)

In [ ]:
# Recipe 1: Normalize line endings to POSIX \n
def normalize_to_posix(text: str) -> str:
    return "\n".join(text.splitlines())

messy_upload = "Key1: Val1\r\nKey2: Val2\rKey3: Val3\n"
clean_posix = normalize_to_posix(messy_upload)
print("Recipe 1 Normalized:", repr(clean_posix))
assert "\r" not in clean_posix

# Recipe 2: Numbered source code formatter
def format_code(code: str, start: int = 1) -> str:
    lines = code.splitlines()
    width = len(str(start + len(lines)))
    return "\n".join(f"{i:>{width}} | {line}" for i, line in enumerate(lines, start=start))

snippet = "def compute(x):\n    return x ** 2"
print("Recipe 2 Code Block:\n" + format_code(snippet))

# Recipe 3: Grouped multiline log parser
def parse_logs(log_data: str) -> list[dict]:
    entries = []
    curr = None
    for line in log_data.splitlines():
        if not line.strip():
            continue
        if line.startswith((" ", "\t")):
            if curr:
                curr["details"].append(line.strip())
        else:
            if curr:
                entries.append(curr)
            curr = {"header": line, "details": []}
    if curr:
        entries.append(curr)
    return entries

log_sample = """2026-10-13 14:00:00 ERROR DB Fail
    at db.py:10
    at conn.py:44
2026-10-13 14:00:01 INFO Restart ok"""

parsed_logs = parse_logs(log_sample)
print("Recipe 3 Parsed Log Count:", len(parsed_logs))
assert len(parsed_logs) == 2
assert len(parsed_logs[0]["details"]) == 2

print("\nAll engineering recipes passed!")

## 5. Production Implementation & Self-Checking Unit Assertions

This production-grade `LineProcessor` class encapsulates clean line extraction, keepends round-trip reconstruction, and paragraph counting.

In [ ]:
class LineProcessor:
    @staticmethod
    def get_clean_lines(text: str) -> list[str]:
        """Splits text and strips surrounding whitespace per line."""
        return [line.strip() for line in text.splitlines() if line.strip()]

    @staticmethod
    def reconstruct_lines(lines_with_breaks: list[str]) -> str:
        """Reassembles lines preserved via keepends=True."""
        return "".join(lines_with_breaks)

    @staticmethod
    def count_paragraphs(text: str) -> int:
        """Counts non-empty lines separated by universal line breaks."""
        return len([ln for ln in text.splitlines() if ln.strip()])


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
processor = LineProcessor()

# 1. Empty string yields empty list
assert "".splitlines() == []
assert "".splitlines(keepends=True) == []

# 2. Trailing newline behavior
assert "Hello\nWorld\n".splitlines() == ["Hello", "World"]
assert "Hello\nWorld\n".split("\n") == ["Hello", "World", ""]

# 3. Universal newlines: CRLF, CR, LF
mixed = "Line1\nLine2\r\nLine3\rLine4"
assert mixed.splitlines() == ["Line1", "Line2", "Line3", "Line4"]

# 4. keepends=True round-trip integrity
lines_kept = mixed.splitlines(keepends=True)
assert lines_kept == ["Line1\n", "Line2\r\n", "Line3\r", "Line4"]
assert processor.reconstruct_lines(lines_kept) == mixed

# 5. Unicode line separator (\u2028)
unicode_lines = "Paragraph1\u2028Paragraph2"
assert unicode_lines.splitlines() == ["Paragraph1", "Paragraph2"]

# 6. Clean line filtering
doc = "  Alpha  \n\n   Beta   \n\n\n  Gamma  \n"
assert processor.get_clean_lines(doc) == ["Alpha", "Beta", "Gamma"]
assert processor.count_paragraphs(doc) == 3

print("Clean lines extracted: ", processor.get_clean_lines(doc))
print("Paragraphs counted:    ", processor.count_paragraphs(doc))
print("All str.splitlines() unit assertions passed successfully!")